In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Literal

from hcc_multimodal.baselines.data import add_rfs_columns, get_hcc_genes
from hcc_multimodal.baselines.transforms import DataType, build_preprocessor
from hcc_multimodal.baselines.evaluation import (
    run_cv_experiment,
    plot_cv_results,
    DeseqCPMSelector,
)
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from itertools import product

In [21]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
RNA_COUNT_FILTER = 15
RNA_COUNT_FILTER_MIN_SAMPLES = 5
DESEQ_PVALUE = 0.1
DESEQ_MIN_FEATURES = 20
RFS_YEARS = [1, 2]
CV_N_FOLDS = 3

SELECTOR: Literal["deseq", "sklearn_kbest"] = "sklearn_kbest"
KBEST: int = 80
SELECTOR_BEFORE_CV: bool = False
PREDEFINED_GENES: bool = True

# Read data

In [15]:
ROOT = Path().resolve().parents[1]


clinical_data = pd.read_csv(
    ROOT
    / "data"
    / "Clinical"
    / "2025_Nov_18_ICL_Resection_Clinical_Outcome_soramic_format.csv"
).dropna(how="all")

rna_data = pd.read_csv(
    ROOT / "data" / "RNA_seq" / "Matrix_output_radiology_only.csv"
).dropna(how="all")

In [16]:
columns = rna_data["Gene Symbol"]
rna_data = rna_data.T.iloc[2:, :]
rna_data.columns = columns.values

# Drop columns with NaN gene symbols and deduplicate gene names
rna_data = rna_data.loc[:, ~pd.isnull(rna_data.columns)]
rna_data = rna_data.loc[:, ~rna_data.columns.duplicated()]

# Convert expression values to numeric, make SID an integer column
rna_data = rna_data.apply(pd.to_numeric, errors="coerce")
rna_data.index = rna_data.index.astype(int)
rna_data = rna_data.rename_axis("SID").reset_index()

print(f"RNA data shape (patients × genes): {rna_data.shape}")
rna_data.head()

RNA data shape (patients × genes): (60, 50987)


,SID,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,...,AC134980.5,AL691520.1,AC139491.7,AC021097.2,AL590381.1,AC003043.2,AC135068.11,AL356417.3,AC104389.6,AP000646.1
0,39,0,0,0,0,0,0,245,0,0,...,0,0,0,0,0,0,0,0,0,0
1,31,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,66,150,0,153,0,0,0,828,63,0,...,0,0,0,0,0,0,0,0,0,0
3,33,139,0,0,0,0,0,3206,0,224,...,0,0,0,0,0,1,0,3,0,0
4,132,0,0,0,0,37,2,985,0,236,...,0,0,0,0,0,1,0,0,0,0


# RFS column

In [17]:
clinical_data = add_rfs_columns(clinical_data)
clinical_data[["RFS_central", "RFS_central_event", "rfs_1year", "rfs_2year"]].head(10)

,RFS_central,RFS_central_event,rfs_1year,rfs_2year
0,31.791781,0.0,0.0,0.0
1,30.279452,1.0,0.0,0.0
2,40.865753,1.0,0.0,0.0
3,14.301370,1.0,0.0,1.0
4,24.756164,1.0,0.0,0.0
5,9.600000,1.0,1.0,1.0
6,8.054795,1.0,1.0,1.0
7,6.936986,1.0,1.0,1.0
8,49.643836,1.0,0.0,0.0
9,11.441096,1.0,1.0,1.0


In [18]:
clinical_data[["RFS_central", "RFS_central_event", "rfs_1year", "rfs_2year"]].tail(10)

,RFS_central,RFS_central_event,rfs_1year,rfs_2year
59,52.734247,0.0,0.0,0.0
60,63.813699,1.0,0.0,0.0
61,75.879452,0.0,0.0,0.0
62,2.367123,0.0,NaN,NaN
63,7.791781,1.0,1.0,1.0
64,5.983562,1.0,1.0,1.0
65,5.786301,1.0,1.0,1.0
66,132.526027,1.0,0.0,0.0
67,27.320548,1.0,0.0,0.0
68,97.347945,0.0,0.0,0.0


In [19]:
print(clinical_data["rfs_1year"].value_counts(dropna=False))
print(clinical_data["rfs_2year"].value_counts(dropna=False))

rfs_1year
0.0    43
1.0    21
NaN     5
Name: count, dtype: int64
rfs_2year
0.0    33
1.0    29
NaN     7
Name: count, dtype: int64


In [22]:
rna_count = rna_data.drop(columns=["SID"])
rna_count.index = rna_data["SID"]
print("Before filtering:", rna_count.shape[1])
rna_count_flt = rna_count.loc[
    :, (rna_count >= RNA_COUNT_FILTER).sum(axis=0) >= RNA_COUNT_FILTER_MIN_SAMPLES
]
print("After filtering:", rna_count_flt.shape[1])

if PREDEFINED_GENES:
    hcc_genes = get_hcc_genes()
    rna_count_flt = rna_count_flt.loc[:, rna_count_flt.columns.intersection(hcc_genes)]
    print("After predefined gene filtering:", rna_count_flt.shape[1])

rna_count_flt.head()

Before filtering: 50986
After filtering: 27991
After predefined gene filtering: 2146


/var/folders/9s/lbf5r70j6_1_c29ygw008t_m0000gn/T/ipykernel_1835/3292440701.py:10: UserWarning: 15 gene(s) not found in RNA-seq matrix and will be excluded: CIDEB, FLJ30679, GABARAPL3, ID2B, LINC02693, MARCHF3, METTL25B, MTARC2, PABIR2, PIGAP1, POU2AF3, PTGR3, SEPTIN4, SEPTIN7, UTP25
  hcc_genes = get_hcc_genes()


,CFH,CYP51A1,RAD52,AOC1,M6PR,ALS2,SLC7A2,ARF5,PDK4,SLC25A13,...,DACH1,HYDIN2,MARCKS,CCL3,PSMB3,SRD5A2,ACACA,MRM1,HIST1H4A,TBCE
SID,,,,,,,,,,,,,,,,,,,,,
39,245,0,0,0,44,70,0,52,0,0,...,95,5,168,0,0,0,0,0,0,0
31,0,0,0,0,0,0,0,0,0,0,...,418,0,0,0,0,0,78,0,0,0
66,828,0,0,1,71,6,0,15,0,10,...,7,211,352,0,0,0,61,0,0,58
33,3206,1,0,1,66,0,121,69,0,147,...,733,0,244,0,1,0,0,0,0,59
132,985,0,0,0,76,0,62,0,481,0,...,8,0,192,62,165,41,0,0,0,0


In [10]:
rna_rfs = rna_count_flt.merge(
    clinical_data[["SID", "rfs_1year", "rfs_2year"]], on="SID"
)
rna_rfs = rna_rfs.set_index("SID")
rna_rfs.head()

,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,AC104162.2,AL033530.1,GABPB1-IT1,BX571818.1,AC093827.5,AC139491.7,AC135068.11,AL356417.3,rfs_1year,rfs_2year
SID,,,,,,,,,,,,,,,,,,,,,
39,0,0,0,0,0,0,245,0,0,0,...,0,0,0,0,0,0,0,0,0.0,0.0
31,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0.0,1.0
66,150,0,153,0,0,0,828,63,0,82,...,123,0,0,0,0,0,0,0,NaN,NaN
33,139,0,0,0,0,0,3206,0,224,0,...,15,0,0,0,64,0,0,3,0.0,1.0
132,0,0,0,0,37,2,985,0,236,0,...,0,0,0,0,0,0,0,0,NaN,NaN


# CV experiments

In [11]:
if SELECTOR == "deseq":
    selector = DeseqCPMSelector(pvalue=DESEQ_PVALUE, min_features=DESEQ_MIN_FEATURES)
elif SELECTOR == "sklearn_kbest":
    selector = SelectKBest(f_classif, k=KBEST)
else:
    raise ValueError(f"Unknown SELECTOR: {SELECTOR!r}")

In [12]:
if SELECTOR == "deseq":
    OUTPUT_DIR = ROOT / "notebooks" / "baselines" / f"{CV_N_FOLDS}_folds"
else:
    OUTPUT_DIR = ROOT / "notebooks" / "baselines" / f"kbest_cpm_{CV_N_FOLDS}_folds"
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
X = rna_rfs.drop(columns=["rfs_1year", "rfs_2year"])
x_columns = {col: DataType.CONTINUOUS for col in X}

selector_first = SELECTOR == "deseq"
rna_cpm = SELECTOR == "sklearn_kbest"

all_records = []

for RFS_YEAR in RFS_YEARS:
    y = rna_rfs[f"rfs_{RFS_YEAR}year"]
    EXPERIMENT_LABEL = f"RFS {RFS_YEAR} year - RNA-seq"

    if SELECTOR_BEFORE_CV:
        y_mask = ~y.isna()
        X_fit, y_fit = X[y_mask], y[y_mask]
        sel_pre = clone(selector)
        if selector_first:
            X_arr = sel_pre.fit_transform(X_fit, y_fit)
            support = (
                sel_pre.support_
                if hasattr(sel_pre, "support_")
                else sel_pre.get_support()
            )
            feature_names = np.asarray(sel_pre.feature_names_in_)[support]
        else:
            prep_pre = build_preprocessor(x_columns, rna_cpm=rna_cpm)
            X_pre = prep_pre.fit_transform(X_fit)
            X_arr = sel_pre.fit_transform(X_pre, y_fit)
            support = sel_pre.get_support()
            preproc_names = np.asarray(prep_pre.get_feature_names_out())
            preproc_names = np.array(
                [n.split("__", 1)[1] if "__" in n else n for n in preproc_names]
            )
            feature_names = preproc_names[support]
        X_cv = pd.DataFrame(np.asarray(X_arr), columns=feature_names, index=X_fit.index)
        y_cv = y_fit
        x_columns_cv = {c: DataType.CONTINUOUS for c in feature_names}
        cv_kwargs = dict(feature_selector=None, selector_first=False, rna_cpm=False)
        print(
            f"[{EXPERIMENT_LABEL}] selector applied before CV: {len(feature_names)} features"
        )
    else:
        X_cv, y_cv, x_columns_cv = X, y, x_columns
        cv_kwargs = dict(
            feature_selector=selector,
            selector_first=selector_first,
            rna_cpm=rna_cpm,
        )

    # --- Logistic Regression ---
    MODELS_LR = {
        f"LR_C={c}": LogisticRegression(
            solver="liblinear",
            l1_ratio=1.0,
            C=c,
            max_iter=1000,
            random_state=RANDOM_STATE,
        )
        for c in [0.001, 0.01, 0.1, 1]
    }
    records_lr, fold_records_lr = run_cv_experiment(
        X_cv,
        y_cv,
        x_columns_cv,
        EXPERIMENT_LABEL,
        models=MODELS_LR,
        n_splits=CV_N_FOLDS,
        **cv_kwargs,
    )
    plot_cv_results(
        fold_records_lr,
        title=f"{EXPERIMENT_LABEL}, Logistic Regression ({SELECTOR})",
        save_path=OUTPUT_DIR / f"rfs_{RFS_YEAR}y_lr.png",
    )

    # --- Random Forest ---
    MODELS_RF = {
        f"RF_max_depth={d}_min_samples_leaf={m}": RandomForestClassifier(
            max_depth=d, min_samples_leaf=m, random_state=RANDOM_STATE
        )
        for d, m in product([2, 4], [5, 10, 15])
    }
    records_rf, fold_records_rf = run_cv_experiment(
        X_cv,
        y_cv,
        x_columns_cv,
        EXPERIMENT_LABEL,
        models=MODELS_RF,
        n_splits=CV_N_FOLDS,
        **cv_kwargs,
    )
    plot_cv_results(
        fold_records_rf,
        title=f"{EXPERIMENT_LABEL}, Random Forest ({SELECTOR})",
        save_path=OUTPUT_DIR / f"rfs_{RFS_YEAR}y_rf.png",
    )

    # Save selected features (one CSV per model type per year, all folds combined)
    for fold_records, model_type in [(fold_records_lr, "lr"), (fold_records_rf, "rf")]:
        feats = []
        for _, row in fold_records.iterrows():
            if row["selected_features"] is not None:
                df = row["selected_features"].copy()
                df["fold"] = row["fold"]
                df["model"] = row["model"]
                feats.append(df)
        if feats:
            (
                pd.concat(feats, ignore_index=True).to_csv(
                    OUTPUT_DIR / f"rfs_{RFS_YEAR}y_{model_type}_selected_features.csv",
                    index=False,
                )
            )

    all_records.extend([records_lr, records_rf])

summary = pd.concat(all_records, ignore_index=True)
display(summary)